In [1]:
import os
import torch
import matplotlib.pyplot as plt

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader


# 设置项目工作目录
project_dir = os.getcwd()
os.chdir(project_dir)

# 设置数据集路径
data_root = "./ml2022spring-hw3b/food11"

train_dir = os.path.join(data_root, "training")
valid_dir = os.path.join(data_root, "validation")
test_dir = os.path.join(data_root, "test")

# 设置训练设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("当前目录：", os.getcwd())
print("训练设备：", device)
print("训练集路径：", train_dir)

当前目录： E:\机器学习\机器学习代码训练\李宏毅2022\HW3
训练设备： cuda
训练集路径： ./ml2022spring-hw3b/food11\training


In [ ]:
#   图片文件夹
# ->Dataset 按需读取
# ->图片预处理
# ->DataLoader 按 batch 取图片
# ->送进模型

# 先来图片预处理

# 训练集预处理
# 训练集做这些操作先模拟坏照片的情况
# 例如 调整尺寸，随机裁剪，随机翻转，随机旋转,这也叫做数据增强
# 它们不会真的保存为新的图片文件，但训练过程看起来像许多不同的版本
# 这样可以减少模型死记硬背照片，减少过拟合
# 标准化和转换为tensor是都要执行的
train_transform=transforms.Compose([
    # 原图是512 x 512 现在把他变成144 x 144
    # 512 x512 太大，训练成本高，这样能减少计算量和显存占用
    transforms.Resize((144,144)),
    # 这一步将144x 144 随机裁剪成128 x128
    transforms.RandomCrop((128,128)),
    # 以百分之五十的概率将图片左右翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 随机旋转照片，角度范围是正负15度
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

# 验证集和测试集的预处理
valid_test_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

# 定义Dataset
import os

# 这里的数据集和前面mnist和cafir-10不同，我们需要自己定义一个Food类
# 因为前面是两个经典的数据集，都设置好了，直接用就行了
# 这里图片的文件名还不一样，得特殊处理一下
class FoodDataset(Dataset):
    def __init__(self,folder,transform = None,is_test = False):
        # 保存图片文件夹位置
        self.folder = folder

        # 保存图片预处理方式
        self.transform = transform

        # 记录当前数据集是否为测试集
        self.is_test = is_test

        # 读取文件夹中所有图片文件
        # sorted 将文件名排序，让图片排序稳定
        # 这里省去了append
        self.files = sorted([
            filename for filename in os.listdir(folder)
            if filename.lower().endswith(('.jpg','.jpeg','.png'))
        ])
    # DataLoader 需要知道数据集长度，因为它要计算大约能组成多少个 batch
    def __len__(self):
        # 返回数据集中的图片数量
        return len(self.files)
    # 负责真正读取图片
    def __getitem__(self,index):
        # 根据index 得到文件名
        filename = self.files[index]

        # 拼接完整图片路径
        image_path=  os.path.join(self.folder,filename)

        # 打开图片，并统一转换为RGB通道
        # 通过完整路径读取硬盘中的图片，得到一个PIL图片对象
        image = Image.open(image_path).convert('RGB')

        # 对图片进行预处理
        if self.transform is not None :
            image = self.transform(image)

        # 测试集文件名没有标签
        if self.is_test:
            label = -1
        else :
            # 例如 ： 7_123.jpg ->标签为7
            label = int(filename.split('_')[0])
        return image,label
# 训练集
train_dataset = FoodDataset(
    folder = train_dir,
    transform = train_transform,
    is_test = False
)
# 验证集
valid_dataset = FoodDataset(
    folder = valid_dir,
    transform = valid_test_transform,
    is_test = False
)
# 测试集
test_dataset = FoodDataset(
    folder = test_dir,
    transform = valid_test_transform,
    is_test = True
)
train_dataset = FoodDataset(
    folder=train_dir,
    transform=train_transform,
    is_test=False
)

valid_dataset = FoodDataset(
    folder=valid_dir,
    transform=valid_test_transform,
    is_test=False
)

test_dataset = FoodDataset(
    folder=test_dir,
    transform=valid_test_transform,
    is_test=True
)

# 设置每一批量的数量
batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    # 训练集需要打乱顺序
    # 验证集和测试集不需要
    shuffle=True,
    # 这里用不上多进程
    num_workers=0,
    # 将CPU内存中的数据更快传到GPU
    pin_memory=torch.cuda.is_available()
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle = False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

images, labels = next(iter(train_loader))

print("一个 batch 的图片形状：", images.shape)
print("一个 batch 的标签形状：", labels.shape)
print("前 10 个标签：", labels[:10])
print("图片数据类型：", images.dtype)
print("标签数据类型：", labels.dtype)

# 先来搭建一个普通CNN
class FoodCNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.features = torch.nn.Sequential(
            # 第一组卷积
            # [64,3,128,128] -> [64,32,64,64]
            torch.nn.Conv2d(
                in_channels = 3,
                out_channels = 32,
                kernel_size = 3,
                padding = 1,
            ),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size = 2),

            # 第二组卷积
            # [64,32,64,64] -> [64,64,32,32]
            torch.nn.Conv2d(
                in_channels = 32,
                out_channels = 64,
                kernel_size = 3,
                padding = 1,
            ),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size = 2),

            # 第三组卷积
            # [64,64,128,128] -> [64,128,16,16]
            torch.nn.Conv2d(
                in_channels = 64,
                out_channels = 128,
                kernel_size = 3,
                padding = 1,
            ),
            torch.nn.BatchNorm2d(128),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size = 2),

            # 第四租卷积
            # [64,128,16,16] ->[64,256,8,8]
            torch.nn.Conv2d(
                in_channels = 128,
                out_channels = 256,
                kernel_size = 3,
                padding = 1,
            ),
            torch.nn.BatchNorm2d(256),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size = 2),

        )
        # 进行平均池化
        # [64,256,8,8] -> [64,256,1,1]
        # 将每个通道里面的 8x8 数值取平均，压缩成一个数字
        # 每张照片有256张特征图，特征图最后压缩成一个数字
        self.avg_pool = torch.nn.AdaptiveAvgPool2d((1, 1))
        # 有11个种类所以最后输出11
        self.classifier = torch.nn.Linear(256,11)

    def forward(self, x):
        x = self.features(x)
        x= self.avg_pool(x)
        # 【64，256，1，1】 -> [64,256]
        x = torch.flatten(x,1)
        x = self.classifier(x)

        return x

model = FoodCNN()
model = model.to(device)
images, labels = next(iter(train_loader))

print("原始图片形状：", images.shape)
print("原始标签形状：", labels.shape)

images = images.to(device)
labels = labels.to(device)

outputs = model(images)

print("模型输出形状：", outputs.shape)
print("模型输出设备：", outputs.device)

# 设置损失函数和优化器
# CrossEntropyLoss 内部已经包含了softmax
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)
num_epochs = 30

# 开始一个完整的训练
train_losses = []
train_accuracies = []
valid_losses = []
valid_accuracies = []

best_valid_acc = 0.0
for epoch in range(num_epochs):
    # 训练阶段
    model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total =0

    for images,labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        # 清空梯度
        optimizer.zero_grad()

        # 向前传播
        outputs = model(images)

        # 计算损失
        loss = criterion(outputs, labels)

        # 反向传播
        loss.backward()

        # 更新模型参数
        optimizer.step()

        # 累计损失
        current_size = images.size(0)
        train_loss_sum += loss.item() * current_size

        # 找到概率最大的
        predictions = outputs.argmax(dim =1)
        train_correct += (predictions == labels).sum().item()
        train_total += current_size

    train_loss  = train_loss_sum / train_total
    train_accuracy = train_correct / train_total

    # 验证阶段
    model.eval()

    valid_loss_sum = 0.0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():
        for images,labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            current_size = images.size(0)
            valid_loss_sum += loss.item() * current_size

            predictions = outputs.argmax(dim =1)
            valid_correct += (predictions == labels).sum().item()
            valid_total += current_size
    valid_loss = valid_loss_sum / valid_total
    valid_accuracy = valid_correct / valid_total

    # 保存每个epoch 的结果
    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)
    valid_losses.append(valid_loss)
    valid_accuracies.append(valid_accuracy)

    # 保存验证集准确率最高的模型
    if valid_accuracy > best_valid_acc:
        best_valid_acc = valid_accuracy

        torch.save(model.state_dict(),'best_model.pth')

    print(
    f"\n训练结束，最佳验证集准确率："f"{best_valid_acc:.4f}")

# 开始画图，看有没有过拟合
epochs = range(1,len(train_losses)+1)

plt.figure(figsize = (8,5))
plt.plot(epochs,train_losses,label = 'Train Loss')
plt.plot(epochs,valid_losses,label = 'Valid Loss')

plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')

plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    [acc * 100 for acc in train_accuracies],
    label="Train Accuracy"
)

plt.plot(
    epochs,
    [acc * 100 for acc in valid_accuracies],
    label="Valid Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training and Validation Accuracy")

plt.legend()
plt.grid(True)
plt.show()

# 加载最佳模型
model.load_state_dict(torch.load(
    'best_model.pth',
     map_location =  device))

#这里要保证模型还是FoodCNN
# 因为这里best_food_cnn.pth只保存参数，不保存FoodCNN结构
model = model.to(device)
model.eval()

print('最佳模型加载成功')

# 用来保存结果
all_predictions = []
model.eval()
with torch.no_grad():
    for images,_ in test_loader:
        images = images.to(device)
        outputs = model(images)
        predictions = outputs.argmax(dim =1)
        # 把预测结果从GPU 搬回CPU,并将tensor转化为普通的python列表
        all_predictions.extend(predictions.cpu().tolist())
# 检查预测和图片数量是否一致
print("测试图片数量：", len(test_dataset))
print("预测结果数量：", len(all_predictions))

import pandas as pd
# 保存为CSV
test_ids = [os.path.splitext(filename)[0] for filename in test_dataset.files]

submisson = pd.DataFrame({})


一个 batch 的图片形状： torch.Size([64, 3, 128, 128])
一个 batch 的标签形状： torch.Size([64])
前 10 个标签： tensor([ 5,  6,  8,  7,  3,  9,  3,  2, 10,  0])
图片数据类型： torch.float32
标签数据类型： torch.int64
原始图片形状： torch.Size([64, 3, 128, 128])
原始标签形状： torch.Size([64])
模型输出形状： torch.Size([64, 11])
模型输出设备： cuda:0

训练结束，最佳验证集准确率：0.3047

训练结束，最佳验证集准确率：0.3434

训练结束，最佳验证集准确率：0.3983
